In [1]:
#=====CELL 1: IMPORTS & SETUP======
# Import all necessary libraries
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, box
import numpy as np
import warnings
from pandas.errors import SettingWithCopyWarning

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore', category=SettingWithCopyWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
#=====CELL 2: DEFINE CONSTANTS & BOUNDING BOX======
# Define the approximate bounding box for Washington County, Utah
# We expanded the MAXY to 37.7 to ensure we catch the northern county data
MINX, MINY, MAXX, MAXY = (-114.1, 37.0, -112.9, 37.7)

# Define a standard Coordinate Reference System (CRS) for the project.
# EPSG:4326 is WGS84 (standard latitude/longitude)
PROJECT_CRS = "EPSG:4326"

print(f"Project CRS set to: {PROJECT_CRS}")
print(f"Washington County Bounding Box defined: {MINX, MINY, MAXX, MAXY}")

Project CRS set to: EPSG:4326
Washington County Bounding Box defined: (-114.1, 37.0, -112.9, 37.7)


In [4]:
#=====CELL 3: LOAD & PRE-FILTER FIRE DATA======
# Load the raw fire data
df_fire_raw = gpd.read_file('/home/caden/dev/school/CS2500-remote/wasiongton_hazards/data/raw/fire/USA-Fire-Area.geojson')

# Re-project to our standard CRS immediately so spatial filtering works
df_fire_raw = df_fire_raw.to_crs(PROJECT_CRS)

# --- Task 5: Filtering (Record-Based) ---
# Filter 1: Keep only rows where the state column is 'UT'
fires_utah = df_fire_raw[df_fire_raw['state'].str.upper() == 'UT'].copy()

# Filter 2: Use a robust polygon intersection to filter for Washington County
washington_bbox_poly = box(MINX, MINY, MAXX, MAXY)
fires_washington = fires_utah[fires_utah.geometry.intersects(washington_bbox_poly)].copy()

print(f"Original fire records: {len(df_fire_raw)}")
print(f"Utah fire records: {len(fires_utah)}")
print(f"Washington Co. fire records: {len(fires_washington)}")

# Sanity Check
if len(fires_washington) == 0:
    print("\n*** WARNING: No fire records found. Check Bounding Box. ***\n")
else:
    print(f"\n*** SUCCESS: Found {len(fires_washington)} fire records. ***\n")

Original fire records: 23776
Utah fire records: 1342
Washington Co. fire records: 211

*** SUCCESS: Found 211 fire records. ***



In [5]:
#=====CELL 4: CLEAN & TRANSFORM FIRE DATA======

# --- Task 5: Filtering (Field-Based) ---
keep_cols_fire = [
    'incidentname', 'fireyear', 'gisacres', 
    'perimeterdatetime', 'agency', 'geometry'
]
fires_clean = fires_washington[keep_cols_fire].copy()

# Rename columns
fires_clean.rename(columns={
    'incidentname': 'name',
    'fireyear': 'year',
    'gisacres': 'acres_burned',
    'perimeterdatetime': 'event_datetime',
    'agency': 'managing_agency'
}, inplace=True)

# --- Part 3: Attribute Data Cleaning ---
fires_clean['event_datetime'] = pd.to_datetime(fires_clean['event_datetime'], errors='coerce')
fires_clean.dropna(subset=['event_datetime', 'geometry'], inplace=True)

# --- Task 1 & 2: Extraction & Derivation ---
fires_clean['event_hour'] = fires_clean['event_datetime'].dt.hour
fires_clean['event_month'] = fires_clean['event_datetime'].dt.month
fires_clean['year'] = fires_clean['year'].astype(int)

# --- Task 4: Binning ---
bins_fire = [0, 100, 1000, 10000, np.inf]
labels_fire = ['Small', 'Medium', 'Large', 'Very Large']
fires_clean['damage_level'] = pd.cut(fires_clean['acres_burned'], bins=bins_fire, labels=labels_fire, right=False)
fires_clean['damage_level'] = fires_clean['damage_level'].cat.add_categories('Minimal').fillna('Minimal')

# --- Part 3: Spatial Cleaning ---
# Fix invalid geometries
invalid_geoms = ~fires_clean.geometry.is_valid
if invalid_geoms.any():
    print(f"Fixing {invalid_geoms.sum()} invalid fire geometries...")
    fires_clean.loc[invalid_geoms, 'geometry'] = fires_clean.loc[invalid_geoms, 'geometry'].buffer(0)

fires_clean['hazard_type'] = 'wildfire'
fires_clean.drop_duplicates(subset=['event_datetime', 'geometry'], inplace=True)

print(f"Fire data cleaned. {len(fires_clean)} records remaining.")

Fixing 6 invalid fire geometries...
Fire data cleaned. 211 records remaining.


In [8]:
#=====CELL 5: LOAD & PRE-FILTER EARTHQUAKE DATA======
# Load the raw earthquake data
df_earthquake_raw = pd.read_csv('../data/raw/earth_quakes/usgs_main.csv')

# Convert to GeoDataFrame
earthquake_gdf = gpd.GeoDataFrame(
    df_earthquake_raw,
    geometry=gpd.points_from_xy(df_earthquake_raw['longitude'], df_earthquake_raw['latitude']),
    crs=PROJECT_CRS
)

# --- Task 5: Filtering ---
# Use robust polygon check
washington_bbox_poly = box(MINX, MINY, MAXX, MAXY)
earthquake_washington = earthquake_gdf[earthquake_gdf.geometry.within(washington_bbox_poly)].copy()

print(f"Original earthquake records: {len(df_earthquake_raw)}")
print(f"Washington Co. earthquake records: {len(earthquake_washington)}")

# Sanity Check
if len(earthquake_washington) == 0:
    print("\n*** WARNING: No earthquake records found. ***\n")
else:
    print(f"\n*** SUCCESS: Found {len(earthquake_washington)} earthquake records. ***\n")

Original earthquake records: 75810
Washington Co. earthquake records: 35

*** SUCCESS: Found 35 earthquake records. ***



In [9]:
#=====CELL 6: CLEAN & TRANSFORM EARTHQUAKE DATA======

# --- Task 5: Filtering (Field-Based) ---
keep_cols_eq = [
    'time', 'latitude', 'longitude', 'depth', 
    'mag', 'place', 'geometry'
]
earthquake_clean = earthquake_washington[keep_cols_eq].copy()

# Rename columns
earthquake_clean.rename(columns={
    'time': 'event_datetime',
    'mag': 'magnitude',
    'depth': 'depth_km',
    'place': 'location_desc'
}, inplace=True)

# --- Part 3: Attribute Cleaning ---
earthquake_clean['event_datetime'] = pd.to_datetime(earthquake_clean['event_datetime'], errors='coerce')
earthquake_clean.dropna(subset=['event_datetime', 'magnitude', 'geometry'], inplace=True)

# --- Task 1: Derivation ---
earthquake_clean['event_month'] = earthquake_clean['event_datetime'].dt.month
earthquake_clean['event_year'] = earthquake_clean['event_datetime'].dt.year

# --- Task 4: Binning ---
bins_eq = [-np.inf, 2.5, 5.4, 6.0, 6.9, np.inf]
labels_eq = ['Minimal', 'Light', 'Moderate', 'Strong', 'Major']
earthquake_clean['danger_level'] = pd.cut(earthquake_clean['magnitude'], bins=bins_eq, labels=labels_eq, right=False)

earthquake_clean['hazard_type'] = 'earthquake'
earthquake_clean.drop_duplicates(subset=['event_datetime', 'magnitude', 'location_desc'], inplace=True)

print(f"Earthquake data cleaned. {len(earthquake_clean)} records remaining.")

Earthquake data cleaned. 35 records remaining.


In [11]:
#=====CELL 7: LOAD & PRE-FILTER FLOOD DATA======
# Load the raw flood data
df_flood_raw = gpd.read_file('../data/raw//flood_data/flood_hazard.geojson')

# Reproject before filtering
df_flood_raw = df_flood_raw.to_crs(PROJECT_CRS)

# --- Task 5: Filtering ---
washington_bbox_poly = box(MINX, MINY, MAXX, MAXY)
flood_washington = df_flood_raw[df_flood_raw.geometry.intersects(washington_bbox_poly)].copy()
# Clip geometries to keep it clean
flood_washington['geometry'] = flood_washington['geometry'].intersection(washington_bbox_poly)

print(f"Original flood zone records: {len(df_flood_raw)}")
print(f"Washington Co. flood zone records: {len(flood_washington)}")

# Sanity Check
if len(flood_washington) == 0:
    print("\n*** WARNING: No flood zones found. ***\n")
else:
    print(f"\n*** SUCCESS: Found {len(flood_washington)} flood zones. ***\n")

Original flood zone records: 13414
Washington Co. flood zone records: 985

*** SUCCESS: Found 985 flood zones. ***



In [12]:
#=====CELL 8: CLEAN & TRANSFORM FLOOD DATA======

# --- Task 5: Filtering ---
keep_cols_flood = ['fld_zone', 'zone_subty', 'depth', 'geometry']
flood_clean = flood_washington[keep_cols_flood].copy()

# --- Part 3: Attribute Cleaning ---
flood_clean['fld_zone'] = flood_clean['fld_zone'].str.upper().str.strip()
flood_clean['zone_subty'] = flood_clean['zone_subty'].fillna('None')

# --- Task 4: Binning ---
bins_flood = [-np.inf, 0, 1, 3, 5, np.inf]
labels_flood = ['Unknown', 'Minimal (0ft)', 'Low (1-3ft)', 'Medium (3-5ft)', 'High (5+ft)']
flood_clean['danger_level'] = pd.cut(flood_clean['depth'], bins=bins_flood, labels=labels_flood, right=True)

# --- Part 3: Spatial Cleaning ---
flood_clean.dropna(subset=['geometry'], inplace=True)
flood_clean = flood_clean[~flood_clean.geometry.is_empty]

invalid_geoms = ~flood_clean.geometry.is_valid
if invalid_geoms.any():
    print(f"Fixing {invalid_geoms.sum()} invalid flood geometries...")
    flood_clean.loc[invalid_geoms, 'geometry'] = flood_clean.loc[invalid_geoms, 'geometry'].buffer(0)

flood_clean['hazard_type'] = 'flood_zone'
flood_clean.drop_duplicates(subset=['fld_zone', 'geometry'], inplace=True)

print(f"Flood data cleaned. {len(flood_clean)} records remaining.")

Flood data cleaned. 985 records remaining.


In [14]:
#=====CELL 9: SAVE CLEANED INDIVIDUAL DATASETS======
# Save the three clean datasets for reference
fires_clean.to_file('../data/processed/washington_fires_clean.geojson', driver='GeoJSON')
earthquake_clean.to_file('../data/processed/washington_earthquakes_clean.geojson', driver='GeoJSON')
flood_clean.to_file('../data/processed/washington_floods_clean.geojson', driver='GeoJSON')
print("Saved all intermediate cleaned files.")

Saved all intermediate cleaned files.


In [15]:
#=====CELL 10: CREATE LOCATION-BASED HAZARD DATASET (SPATIAL JOIN)======
# --- Task 3, Task 6, & Part 2 (Data Blending) ---

# Define a Projected CRS (UTM Zone 12N for Utah meters) and Buffer Size (5km)
UTM_CRS = "EPSG:26912" 
BUFFER_METERS = 5000
print(f"Using projected CRS: {UTM_CRS} with a {BUFFER_METERS}m buffer.")

# Step 1: Create a grid of points (0.01 deg ~ 1.1km)
x = np.arange(MINX, MAXX, 0.01)
y = np.arange(MINY, MAXY, 0.01)
x_coords, y_coords = np.meshgrid(x, y)
points = [Point(x, y) for x, y in zip(x_coords.flatten(), y_coords.flatten())]
danger_grid = gpd.GeoDataFrame(geometry=points, crs=PROJECT_CRS)
print(f"Created a danger grid with {len(danger_grid)} points.")

# Project all datasets to UTM for accurate joining
print("Projecting all datasets to UTM CRS...")
danger_grid_proj = danger_grid.to_crs(UTM_CRS)
fires_proj = fires_clean.to_crs(UTM_CRS)
earthquakes_proj = earthquake_clean.to_crs(UTM_CRS)
flood_proj = flood_clean.to_crs(UTM_CRS)

# Step 2: Join with Flood Data (Projected)
grid_with_flood = gpd.sjoin(danger_grid_proj, flood_proj, how='left', predicate='within')
grid_with_flood.rename(columns={'fld_zone': 'flood_zone', 'danger_level': 'flood_danger'}, inplace=True)
grid_with_flood = grid_with_flood[['flood_zone', 'flood_danger']]
print("Spatially joined grid with flood zones.")

# Step 3: Buffer Grid and Join with Events
danger_grid_buffered_proj = danger_grid_proj.buffer(BUFFER_METERS)
print(f"Buffered grid points by {BUFFER_METERS} meters.")

# Join Fires
fires_sjoined = gpd.sjoin(
    gpd.GeoDataFrame(geometry=danger_grid_buffered_proj, crs=UTM_CRS), 
    fires_proj, how='left', predicate='intersects'
)
fire_counts = fires_sjoined.groupby(fires_sjoined.index)['index_right'].count()
fire_avg_acres = fires_sjoined.groupby(fires_sjoined.index)['acres_burned'].mean()
print("Aggregated fire counts per grid cell.")

# Join Earthquakes
earthquakes_sjoined = gpd.sjoin(
    gpd.GeoDataFrame(geometry=danger_grid_buffered_proj, crs=UTM_CRS),
    earthquakes_proj, how='left', predicate='intersects'
)
eq_counts = earthquakes_sjoined.groupby(earthquakes_sjoined.index)['index_right'].count()
eq_max_mag = earthquakes_sjoined.groupby(earthquakes_sjoined.index)['magnitude'].max()
print("Aggregated earthquake counts per grid cell.")

# Step 4: Combine back to original Lat/Lon Grid
danger_grid['fire_count'] = fire_counts
danger_grid['fire_avg_acres'] = fire_avg_acres
danger_grid['earthquake_count'] = eq_counts
danger_grid['earthquake_max_mag'] = eq_max_mag

final_hazards_by_location = danger_grid.merge(
    grid_with_flood, left_index=True, right_index=True, how='left'
)

# Fill NaNs
final_hazards_by_location['fire_count'].fillna(0, inplace=True)
final_hazards_by_location['fire_avg_acres'].fillna(0, inplace=True)
final_hazards_by_location['earthquake_count'].fillna(0, inplace=True)
final_hazards_by_location['earthquake_max_mag'].fillna(0, inplace=True)

# Handle Categorical Filling safely
if not isinstance(final_hazards_by_location['flood_danger'].dtype, pd.CategoricalDtype):
     final_hazards_by_location['flood_danger'] = pd.Categorical(
         final_hazards_by_location['flood_danger'], 
         categories=flood_clean['danger_level'].cat.categories
     )

final_hazards_by_location['flood_zone'].fillna('None', inplace=True)
final_hazards_by_location['flood_danger'] = final_hazards_by_location['flood_danger'].cat.add_categories('None')
final_hazards_by_location['flood_danger'].fillna('None', inplace=True)

print("\nFinal location-based hazard dataset created.")

Using projected CRS: EPSG:26912 with a 5000m buffer.
Created a danger grid with 8520 points.
Projecting all datasets to UTM CRS...
Spatially joined grid with flood zones.
Buffered grid points by 5000 meters.
Aggregated fire counts per grid cell.
Aggregated earthquake counts per grid cell.

Final location-based hazard dataset created.


In [17]:
#=====CELL 11: SAVE FINAL COMBINED DATASET======
final_hazards_by_location.to_file(
    '../data/processed/hazards_by_location_washington.geojson', 
    driver='GeoJSON'
)
print(f"Successfully saved final dataset with {len(final_hazards_by_location)} grid points.")

Successfully saved final dataset with 8520 grid points.


In [18]:
#=====CELL 12: VERIFY JOIN RESULTS (SANITY CHECK)======
print("--- Sanity Check for Hazard Joins ---")
print(f"Total fire count (sum): {final_hazards_by_location['fire_count'].sum()}")
print(f"Highest 'fire_count' in one cell: {final_hazards_by_location['fire_count'].max()}")
print(f"Total earthquake count (sum): {final_hazards_by_location['earthquake_count'].sum()}")
print(f"Max magnitude found: {final_hazards_by_location['earthquake_max_mag'].max()}")
print("Flood zone distribution:")
print(final_hazards_by_location['flood_zone'].value_counts().head())

--- Sanity Check for Hazard Joins ---
Total fire count (sum): 27465
Highest 'fire_count' in one cell: 18
Total earthquake count (sum): 2676
Max magnitude found: 2.24
Flood zone distribution:
flood_zone
X       6265
None    2203
A         40
AE        12
Name: count, dtype: int64


In [19]:
#=====CELL 13: CREATE FINAL 'DANGER_LEVEL'======
# Revised scoring based on your actual data ranges
print("\n--- Creating Final Danger Level ---")

final_hazards_by_location['danger_score'] = 0

# Fire Scoring
final_hazards_by_location.loc[final_hazards_by_location['fire_count'] > 0, 'danger_score'] += 1
final_hazards_by_location.loc[final_hazards_by_location['fire_count'] > 5, 'danger_score'] += 1

# Earthquake Scoring (Threshold adjusted to 2.0 to catch actual data)
final_hazards_by_location.loc[final_hazards_by_location['earthquake_count'] > 0, 'danger_score'] += 1
final_hazards_by_location.loc[final_hazards_by_location['earthquake_max_mag'] > 2.0, 'danger_score'] += 1

# Flood Scoring
final_hazards_by_location.loc[final_hazards_by_location['flood_zone'] != 'None', 'danger_score'] += 1
final_hazards_by_location.loc[final_hazards_by_location['flood_zone'].isin(['A', 'AE', 'AO']), 'danger_score'] += 1
final_hazards_by_location.loc[final_hazards_by_location['flood_danger'].isin(['Medium (3-5ft)', 'High (5+ft)']), 'danger_score'] += 1

# Binning the final score
bins = [-1, 0, 1, 2, 3, np.inf] 
labels = ['Low', 'Moderate', 'High', 'Very High', 'Extreme']
final_hazards_by_location['danger_level'] = pd.cut(
    final_hazards_by_location['danger_score'], 
    bins=bins, 
    labels=labels
)

print("Final Danger Level Distribution:")
print(final_hazards_by_location['danger_level'].value_counts())

print("\n--- Top 5 Most Dangerous Locations ---")
print(final_hazards_by_location.sort_values(by='danger_score', ascending=False).head())


--- Creating Final Danger Level ---
Final Danger Level Distribution:
danger_level
High         3492
Very High    2116
Moderate     1557
Low           907
Extreme       448
Name: count, dtype: int64

--- Top 5 Most Dangerous Locations ---
                   geometry  fire_count  fire_avg_acres  earthquake_count  \
3084  POINT (-113.26 37.25)           6     1176.511485                 1   
3195  POINT (-113.35 37.26)           7     2009.365614                 1   
3074  POINT (-113.36 37.25)           6     2343.143053                 1   
2955  POINT (-113.35 37.24)           6     2343.143053                 1   
2060   POINT (-113.9 37.17)           8    13990.946866                 1   

      earthquake_max_mag flood_zone flood_danger  danger_score danger_level  
3084                0.79          A      Unknown             5      Extreme  
3195                1.50          A      Unknown             5      Extreme  
3074                1.50          A      Unknown             5  

In [23]:
#=====CELL 14: LOAD PROCESSED DATA FOR ANALYSIS======
# Load the final dataset created in the previous checkpoint
import pandas as pd
import geopandas as gpd

final_hazards_by_location.to_file('../data/processed/hazards_by_location_washington.geojson', driver='GeoJSON')
print("File updated with Danger Levels.")

# Load the combined hazard grid
df_analysis = gpd.read_file('../data/processed/hazards_by_location_washington.geojson')

# Ensure the categorical order is respected for the danger level
danger_order = ['Low', 'Moderate', 'High', 'Very High', 'Extreme']
df_analysis['danger_level'] = pd.Categorical(
    df_analysis['danger_level'], 
    categories=danger_order, 
    ordered=True
)

print(f"Data loaded for analysis: {len(df_analysis)} records.")
print(df_analysis.info())

File updated with Danger Levels.
Data loaded for analysis: 8520 records.
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 8520 entries, 0 to 8519
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   fire_count          8520 non-null   int32   
 1   fire_avg_acres      8520 non-null   float64 
 2   earthquake_count    8520 non-null   int32   
 3   earthquake_max_mag  8520 non-null   float64 
 4   flood_zone          8520 non-null   object  
 5   flood_danger        8520 non-null   object  
 6   danger_score        8520 non-null   int32   
 7   danger_level        8520 non-null   category
 8   geometry            8520 non-null   geometry
dtypes: category(1), float64(2), geometry(1), int32(3), object(2)
memory usage: 441.3+ KB
None


In [24]:
#=====CELL 15: RQ3 ANALYSIS - RISK DISTRIBUTION======
# Research Question: What is the overall distribution of risk levels across the county?

print("--- RQ3: Hazard Risk Distribution Analysis ---")

# Calculate raw counts
risk_counts = df_analysis['danger_level'].value_counts().sort_index()

# Calculate percentages
risk_percents = df_analysis['danger_level'].value_counts(normalize=True).sort_index() * 100

# Combine into a summary table
dist_summary = pd.DataFrame({
    'Grid_Point_Count': risk_counts,
    'Percentage_of_County': risk_percents.round(2)
})

print("\nDistribution of Danger Levels:")
print(dist_summary)

# Calculate basic descriptive stats for the numeric score
print("\nDescriptive Statistics for Danger Score:")
print(df_analysis['danger_score'].describe())

--- RQ3: Hazard Risk Distribution Analysis ---

Distribution of Danger Levels:
              Grid_Point_Count  Percentage_of_County
danger_level                                        
Low                        907                 10.65
Moderate                  1557                 18.27
High                      3492                 40.99
Very High                 2116                 24.84
Extreme                    448                  5.26

Descriptive Statistics for Danger Score:
count    8520.000000
mean        1.958333
std         1.033436
min         0.000000
25%         1.000000
50%         2.000000
75%         3.000000
max         5.000000
Name: danger_score, dtype: float64


In [25]:
#=====CELL 16: RQ2 ANALYSIS - FIRE vs FLOOD RELATIONSHIP======
# Research Question: Is there a spatial correlation between wildfire frequency and specific flood zones?

print("--- RQ2: Fire Frequency by Flood Zone Analysis ---")

# We group by flood zone and look at fire statistics
# We include 'count' to see how many grid points are in that zone (context)
# We include 'mean' to see the average fire risk per zone type
fire_flood_stats = df_analysis.groupby('flood_zone')['fire_count'].agg(['count', 'sum', 'mean'])

# Rename columns for clarity
fire_flood_stats.rename(columns={
    'count': 'Total_Grid_Points',
    'sum': 'Total_Fires_Observed',
    'mean': 'Avg_Fires_Per_Point'
}, inplace=True)

# Sort by average fire frequency
fire_flood_stats = fire_flood_stats.sort_values(by='Avg_Fires_Per_Point', ascending=False)

print("\nWildfire Statistics separated by Flood Zone:")
print(fire_flood_stats)

--- RQ2: Fire Frequency by Flood Zone Analysis ---

Wildfire Statistics separated by Flood Zone:
            Total_Grid_Points  Total_Fires_Observed  Avg_Fires_Per_Point
flood_zone                                                              
A                          40                   170             4.250000
X                        6265                 25484             4.067678
AE                         12                    37             3.083333
None                     2203                  1774             0.805266


In [26]:
#=====CELL 17: RQ1 ANALYSIS - IDENTIFYING HIGH RISK LOCATIONS======
# Research Question: What specific locations face the highest compound risk?

print("--- RQ1: Top 10 Most Dangerous Locations ---")

# Sort by danger score descending
top_risk_locs = df_analysis.sort_values(by='danger_score', ascending=False).head(10)

# Select only readable columns for the report
report_cols = [
    'danger_score', 
    'danger_level', 
    'fire_count', 
    'earthquake_max_mag', 
    'flood_zone',
    'geometry' # useful if you want to grab coordinates
]

print("\nTop 10 Highest Risk Grid Points:")
print(top_risk_locs[report_cols])

# Optional: Get the centroid coordinates of the #1 most dangerous spot
most_dangerous = top_risk_locs.iloc[0]
coords = most_dangerous.geometry.centroid
print(f"\nMost Dangerous Coordinate: Lat {coords.y:.4f}, Lon {coords.x:.4f}")
print(f"Reason: {most_dangerous['fire_count']} fires, Magnitude {most_dangerous['earthquake_max_mag']} quake risk, Flood Zone {most_dangerous['flood_zone']}")

--- RQ1: Top 10 Most Dangerous Locations ---

Top 10 Highest Risk Grid Points:
      danger_score danger_level  fire_count  earthquake_max_mag flood_zone  \
3084             5      Extreme           6                0.79          A   
3195             5      Extreme           7                1.50          A   
3074             5      Extreme           6                1.50          A   
2955             5      Extreme           6                1.50          A   
2060             4      Extreme           8                0.20          X   
3507             4      Extreme           6                0.20          X   
3508             4      Extreme           6                0.20          X   
2061             4      Extreme           7                0.20          X   
2062             4      Extreme          11                0.20          X   
2063             4      Extreme          12                0.20          X   

                   geometry  
3084  POINT (-113.26 37.25)  
31